# 04 — XGBoost Optuna Tuning
## Hackathon IBM — Détection de Fraude Bancaire

**Objectif** : construire le modèle **XGBoost ultime** — profond, régularisé — en optimisant les hyperparamètres avec **Optuna** sur **4× NVIDIA T4**.

### Workflow

1. **Préparation** du dataset `train_original` via `preprocess_pipeline.preprocess_dataset` (même pipeline que notebook 02) → sauvegarde `prepared_train_original.parquet`.
2. **Recherche Optuna** (50-100 trials) sur `prepared_train_005.0_pct.parquet` avec split interne 80/20 + early stopping.
   - Métrique optimisée : **PR-AUC** (robuste au déséquilibre).
   - Espace de recherche large : modèles profonds, `scale_pos_weight` jusqu'à 500.
   - Exécution **parallèle sur les 4 GPUs** (1 trial par GPU).
3. **Entraînement final** avec les meilleurs hyperparamètres sur **2 datasets** :
   - `prepared_train_005.0_pct.parquet` (undersamplé)
   - `prepared_train_original.parquet` (distribution originale)
4. **Évaluation** sur `prepared_test_050.0_pct.parquet` + log CSV + sauvegarde des modèles.
5. **Visualisations** : historique d'optimisation, importance des hyperparamètres, comparaison finale.

---
## 0. Installation (une seule fois)

In [ ]:
# !pip install -q xgboost optuna optuna-integration scikit-learn pandas numpy pyarrow matplotlib plotly

---
## 1. Imports & détection du matériel

In [ ]:
import os
import gc
import time
import json
import threading
import warnings
from pathlib import Path
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from optuna.pruners  import MedianPruner

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, recall_score, precision_score, accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

# Import du pipeline de préparation (même que notebook 02)
from preprocess_pipeline import preprocess_dataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)

# --- CUDA detection ---
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    N_GPUS = torch.cuda.device_count() if CUDA_AVAILABLE else 0
except ImportError:
    # torch n'est pas requis pour XGBoost GPU, mais pratique pour compter les devices
    CUDA_AVAILABLE = False
    N_GPUS = 0

XGB_VERSION = tuple(int(x) for x in xgb.__version__.split('.')[:2])

print('=== Environment ===')
print(f'xgboost  : {xgb.__version__}')
print(f'optuna   : {optuna.__version__}')
print(f'CUDA     : {CUDA_AVAILABLE} (devices={N_GPUS})')
if CUDA_AVAILABLE:
    import torch
    for i in range(N_GPUS):
        print(f'  GPU[{i}] : {torch.cuda.get_device_name(i)}')

---
## 2. Configuration globale

In [ ]:
DATA_DIR    = Path('.')
LOG_DIR     = Path('logs_training')
MODELS_DIR  = LOG_DIR / 'models'
OPTUNA_DIR  = LOG_DIR / 'optuna'

for d in [LOG_DIR, MODELS_DIR, OPTUNA_DIR]:
    d.mkdir(exist_ok=True, parents=True)

RESULTS_CSV = Path('logs_xgboost_optuna_results.csv')
STUDY_PATH  = OPTUNA_DIR / 'xgboost_study.pkl'

# --- Fichiers utilisés ---
# Le dataset brut avec la distribution originale (à fournir par l'utilisateur)
ORIGINAL_CANDIDATES = ['train_original.parquet', 'train_original.csv']
# Le dataset undersamplé à 5% pour la recherche Optuna (doit déjà exister)
TUNING_DATA = 'prepared_train_005.0_pct.parquet'
# Le dataset de test fixe
TEST_DATA   = 'prepared_test_050.0_pct.parquet'
# Le dataset préparé avec la distribution originale (produit à l'étape 1)
PREPARED_ORIGINAL = 'prepared_train_original.parquet'

TARGET = 'is_fraud'
RANDOM_STATE = 42

# --- Paramètres Optuna ---
N_TRIALS           = 80            # entre 50 et 100 selon le temps dispo
OPTUNA_TIMEOUT_MIN = 60            # temps max en minutes (sécurité)
OPTUNA_N_JOBS      = max(1, N_GPUS) if CUDA_AVAILABLE else 1
OPTIMIZE_METRIC    = 'PR-AUC'      # 'PR-AUC' ou 'F1' — on maximise

# --- Budget d'entraînement final (plus élevé que la recherche) ---
FINAL_N_ESTIMATORS     = 3000
FINAL_EARLY_STOPPING   = 150

np.random.seed(RANDOM_STATE)

print(f'Trials Optuna    : {N_TRIALS}')
print(f'Parallel jobs    : {OPTUNA_N_JOBS}  (1 trial par GPU si parallèle)')
print(f'Métrique         : {OPTIMIZE_METRIC} (maximize)')
print(f'Timeout Optuna   : {OPTUNA_TIMEOUT_MIN} min')
print(f'Results CSV      : {RESULTS_CSV.resolve()}')
print(f'Models dir       : {MODELS_DIR.resolve()}')

---
## 3. ÉTAPE 1 — Préparation du dataset `train_original`

On détecte automatiquement le format (CSV ou Parquet). On applique exactement le même pipeline que dans le notebook 02 (via `preprocess_pipeline.preprocess_dataset`).

In [ ]:
def find_original_file() -> Path:
    """Cherche le fichier brut train_original.{parquet,csv} dans le répertoire courant."""
    for name in ORIGINAL_CANDIDATES:
        p = DATA_DIR / name
        if p.exists():
            return p
    raise FileNotFoundError(
        f'Aucun fichier original trouvé. Attendu : un de {ORIGINAL_CANDIDATES} '
        f'dans {DATA_DIR.resolve()}'
    )


def load_raw_file(path: Path) -> pd.DataFrame:
    """Lit un CSV ou Parquet indifféremment."""
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path)


# Si le fichier préparé existe déjà, on ne refait pas le boulot
prepared_original_path = DATA_DIR / PREPARED_ORIGINAL

if prepared_original_path.exists():
    print(f'✓ {PREPARED_ORIGINAL} existe déjà — on skip la préparation.')
else:
    raw_path = find_original_file()
    print(f'Chargement    : {raw_path.name}')
    df_raw = load_raw_file(raw_path)
    print(f'  shape raw    : {df_raw.shape}')
    print(f'  mémoire raw  : {df_raw.memory_usage(deep=True).sum()/1024**2:.1f} MB')

    print('\nApplication du pipeline preprocess_dataset …')
    df_prep = preprocess_dataset(df_raw, verbose=True)
    print(f'\n  shape prep   : {df_prep.shape}')
    print(f'  taux fraude  : {df_prep[TARGET].mean()*100:.4f} %')

    df_prep.to_parquet(prepared_original_path, index=False, compression='snappy')
    print(f'\n✓ Sauvegardé : {prepared_original_path}')

    del df_raw, df_prep
    gc.collect()

---
## 4. Helpers (chargement, alignement des catégories, métriques)

In [ ]:
def load_dataset(path) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """Charge un .parquet préparé. Retourne (X, y, cat_feature_names)."""
    df = pd.read_parquet(path)
    y = df[TARGET].astype(np.int8)
    X = df.drop(columns=[TARGET])
    cat_features = [c for c, dt in X.dtypes.items() if str(dt) == 'category']
    return X, y, cat_features


def align_categories(X_train: pd.DataFrame, X_test: pd.DataFrame,
                     cat_features: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Force les mêmes catégories dans train et test (union des modalités).
    Requis par XGBoost quand enable_categorical=True."""
    X_train = X_train.copy()
    X_test  = X_test.copy()
    for c in cat_features:
        all_cats = (
            pd.api.types.union_categoricals([X_train[c], X_test[c]], sort_categories=True)
              .categories
        )
        X_train[c] = pd.Categorical(X_train[c], categories=all_cats)
        X_test[c]  = pd.Categorical(X_test[c],  categories=all_cats)
    return X_train, X_test


def compute_metrics(y_true, y_pred, y_proba) -> Dict[str, float]:
    metrics = {
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Accuracy':  accuracy_score(y_true, y_pred),
        'PR-AUC':    average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        'ROC-AUC':   roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
    }
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0
    metrics.update({'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)})
    return metrics


def xgb_device_for_trial(trial_number: int) -> str:
    """Round-robin des GPUs pour les trials Optuna en parallèle.
    Retourne 'cuda:{i}' ou 'cpu' selon la dispo."""
    if not CUDA_AVAILABLE or N_GPUS == 0:
        return 'cpu'
    return f'cuda:{trial_number % N_GPUS}'

---
## 5. Chargement des données pour la recherche Optuna

On charge :
- `prepared_train_005.0_pct.parquet` → split **80/20 stratifié** interne (20 % pour l'early stopping de chaque trial).
- `prepared_test_050.0_pct.parquet` → test set fixe pour l'évaluation finale (**pas utilisé par Optuna**).

In [ ]:
# --- Dataset de tuning (5%) ---
tuning_path = DATA_DIR / TUNING_DATA
X_tune_full, y_tune_full, cat_features = load_dataset(tuning_path)
print(f'Tuning dataset : {tuning_path.name}')
print(f'  shape   : {X_tune_full.shape}')
print(f'  fraud   : {y_tune_full.mean()*100:.3f} %')
print(f'  cat cols: {len(cat_features)} -> {cat_features}')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_tune_full, y_tune_full,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_tune_full,
)

# Alignement des catégories (indispensable pour enable_categorical=True)
X_tr, X_val = align_categories(X_tr, X_val, cat_features)

print(f'\n  train : {X_tr.shape}  (fraud={y_tr.mean()*100:.3f} %)')
print(f'  val   : {X_val.shape}  (fraud={y_val.mean()*100:.3f} %)')

# --- Test set pour l'évaluation finale ---
test_path = DATA_DIR / TEST_DATA
X_test, y_test, _ = load_dataset(test_path)
print(f'\nTest dataset   : {test_path.name}')
print(f'  shape   : {X_test.shape}')
print(f'  fraud   : {y_test.mean()*100:.3f} %  ({int(y_test.sum())} positifs)')

---
## 6. ÉTAPE 2 — Fonction objectif Optuna

**Espace de recherche** (demande du cahier des charges) :

| Hyperparamètre | Range | Type |
|---|---|---|
| `n_estimators` | 500 – 2000 | int |
| `max_depth` | 6 – 14 | int |
| `learning_rate` | 0.005 – 0.1 | log-uniform |
| `subsample` | 0.5 – 1.0 | uniform |
| `colsample_bytree` | 0.5 – 1.0 | uniform |
| `scale_pos_weight` | 10 – 500 | log-uniform |
| `gamma` | 0.0 – 5.0 | uniform (anti-overfitting) |
| `min_child_weight` | 1 – 20 | int (anti-overfitting) |
| `reg_lambda` | 1e-3 – 10 | log-uniform |
| `reg_alpha` | 1e-3 – 10 | log-uniform |

Early stopping à `early_stopping_rounds=50` → on ne consomme réellement que `best_iteration` arbres pour chaque trial.

In [ ]:
def build_xgb_params(trial: optuna.Trial, device: str) -> dict:
    """Construit le dict d'hyperparamètres XGBoost à partir du trial Optuna."""
    params = dict(
        n_estimators     = trial.suggest_int('n_estimators', 500, 2000, step=50),
        max_depth        = trial.suggest_int('max_depth', 6, 14),
        learning_rate    = trial.suggest_float('learning_rate', 5e-3, 1e-1, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        scale_pos_weight = trial.suggest_float('scale_pos_weight', 10.0, 500.0, log=True),
        gamma            = trial.suggest_float('gamma', 0.0, 5.0),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 20),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        reg_alpha        = trial.suggest_float('reg_alpha',  1e-3, 10.0, log=True),
        # --- Fixes ---
        objective        = 'binary:logistic',
        eval_metric      = 'aucpr',
        random_state     = RANDOM_STATE,
        enable_categorical = True,
        n_jobs           = -1,
        verbosity        = 0,
        early_stopping_rounds = 50,
    )
    # Configuration GPU / CPU
    if device.startswith('cuda'):
        if XGB_VERSION >= (2, 0):
            params.update(dict(tree_method='hist', device=device))
        else:
            gpu_id = int(device.split(':')[-1]) if ':' in device else 0
            params.update(dict(tree_method='gpu_hist', gpu_id=gpu_id,
                               predictor='gpu_predictor'))
    else:
        params.update(dict(tree_method='hist'))
    return params


def objective(trial: optuna.Trial) -> float:
    """Entraîne XGBoost sur (X_tr, y_tr) avec early stopping sur (X_val, y_val).
    Retourne la métrique à maximiser, calculée sur X_val."""

    device = xgb_device_for_trial(trial.number)
    params = build_xgb_params(trial, device)

    model = xgb.XGBClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False,
    )

    y_proba = model.predict_proba(X_val)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)

    # Métrique principale
    if OPTIMIZE_METRIC == 'F1':
        score = f1_score(y_val, y_pred, zero_division=0)
    else:  # PR-AUC
        score = average_precision_score(y_val, y_proba)

    # Logging auxiliaire pour enrichir le study (consultable dans les visualisations)
    trial.set_user_attr('best_iteration', int(getattr(model, 'best_iteration', params['n_estimators'])))
    trial.set_user_attr('val_f1',     f1_score(y_val, y_pred, zero_division=0))
    trial.set_user_attr('val_recall', recall_score(y_val, y_pred, zero_division=0))
    trial.set_user_attr('val_prauc',  average_precision_score(y_val, y_proba))
    trial.set_user_attr('device',     device)

    del model
    gc.collect()
    return float(score)

---
## 7. Lancement de l'étude Optuna

- **Sampler TPE** (Tree-structured Parzen Estimator) : bien plus efficace que la grille aléatoire.
- **Median Pruner** : coupe les trials qui sous-performent tôt pour gagner du temps.
- **`n_jobs = N_GPUS`** : les trials tournent en parallèle, chacun sur un GPU distinct via `xgb_device_for_trial()`.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

sampler = TPESampler(seed=RANDOM_STATE, multivariate=True)
pruner  = MedianPruner(n_startup_trials=10, n_warmup_steps=5)

study = optuna.create_study(
    study_name='xgboost_fraud_tuning',
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
)

print(f'Démarrage : {N_TRIALS} trials, {OPTUNA_N_JOBS} en parallèle, '
      f'timeout={OPTUNA_TIMEOUT_MIN} min, métrique={OPTIMIZE_METRIC}')
print('—' * 90)

t0 = time.perf_counter()
study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=OPTUNA_N_JOBS,
    timeout=OPTUNA_TIMEOUT_MIN * 60,
    show_progress_bar=True,
    gc_after_trial=True,
)
elapsed = time.perf_counter() - t0

print('—' * 90)
print(f'Optuna terminé en {elapsed/60:.1f} min | {len(study.trials)} trials effectués')
print(f'\nMeilleur {OPTIMIZE_METRIC} (val) : {study.best_value:.6f}')
print(f'\nMeilleurs hyperparamètres :')
for k, v in study.best_params.items():
    print(f'  {k:<22} = {v}')

# Persistance de l'étude (rechargeable avec joblib.load)
import joblib
joblib.dump(study, STUDY_PATH)
print(f'\nStudy sauvegardée : {STUDY_PATH}')

---
## 8. Visualisations Optuna

In [ ]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

fig1 = plot_optimization_history(study)
fig1.update_layout(title=f'Optimization history — {OPTIMIZE_METRIC} (val)')
fig1.write_html(str(OPTUNA_DIR / 'optimization_history.html'))
fig1.show()

fig2 = plot_param_importances(study)
fig2.update_layout(title='Hyperparameter importances')
fig2.write_html(str(OPTUNA_DIR / 'param_importances.html'))
fig2.show()

# Bonus : coordonnées parallèles — utile pour voir les interactions entre hyperparams
try:
    fig3 = plot_parallel_coordinate(study,
                                    params=['max_depth', 'learning_rate',
                                            'scale_pos_weight', 'subsample',
                                            'colsample_bytree', 'min_child_weight'])
    fig3.update_layout(title='Parallel coordinate plot (top params)')
    fig3.write_html(str(OPTUNA_DIR / 'parallel_coordinate.html'))
    fig3.show()
except Exception as e:
    print(f'(parallel coordinate plot non généré : {e})')

print(f'\nVisualisations HTML sauvegardées dans : {OPTUNA_DIR.resolve()}')

In [ ]:
# DataFrame synthétique des trials (pratique pour inspecter)
trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs', 'state'))
trials_df = trials_df.sort_values('value', ascending=False).reset_index(drop=True)
print('=== TOP 10 trials ===')
display_cols = ['number', 'value', 'user_attrs_val_f1', 'user_attrs_val_recall',
                'user_attrs_best_iteration']
display_cols = [c for c in display_cols if c in trials_df.columns]
print(trials_df[display_cols].head(10).to_string(index=False))

trials_df.to_csv(OPTUNA_DIR / 'trials.csv', index=False)

---
## 9. ÉTAPE 3 — Entraînement final sur les 2 datasets

On reprend les meilleurs hyperparamètres trouvés par Optuna, on augmente `n_estimators` à **3000** avec `early_stopping_rounds=150` pour un entraînement final plus soigneux.

**Deux modèles** entraînés avec les **mêmes** hyperparamètres :
- `xgboost_optuna_005`      → sur `prepared_train_005.0_pct.parquet`
- `xgboost_optuna_original` → sur `prepared_train_original.parquet`

Les deux sont évalués sur `prepared_test_050.0_pct.parquet`.

In [ ]:
def build_final_xgb_params(best_params: dict) -> dict:
    """Merge les best_params Optuna avec un budget d'entraînement étendu."""
    params = dict(best_params)
    params.update(dict(
        n_estimators=FINAL_N_ESTIMATORS,
        objective='binary:logistic',
        eval_metric='aucpr',
        random_state=RANDOM_STATE,
        enable_categorical=True,
        n_jobs=-1,
        verbosity=0,
        early_stopping_rounds=FINAL_EARLY_STOPPING,
    ))
    # GPU 0 pour les runs finaux (on n'a plus besoin de parallélisme)
    if CUDA_AVAILABLE:
        if XGB_VERSION >= (2, 0):
            params.update(dict(tree_method='hist', device='cuda:0'))
        else:
            params.update(dict(tree_method='gpu_hist', gpu_id=0))
    else:
        params.update(dict(tree_method='hist'))
    return params


def train_final_model(train_path: Path, best_params: dict, tag: str) -> dict:
    """Entraîne XGBoost avec best_params sur train_path, évalue sur (X_test, y_test).
    Sauvegarde le modèle en .ubj et retourne un dict de métriques."""

    X_full, y_full, cat_feats = load_dataset(train_path)

    # Split interne pour early stopping — stratifié sur y
    X_tr_f, X_val_f, y_tr_f, y_val_f = train_test_split(
        X_full, y_full, test_size=0.15, random_state=RANDOM_STATE, stratify=y_full
    )
    # Aligne ces deux + le test avec l'union globale des catégories
    X_tr_f, X_val_f = align_categories(X_tr_f, X_val_f, cat_feats)
    X_tr_f, X_test_al = align_categories(X_tr_f, X_test, cat_feats)
    X_val_f, _        = align_categories(X_val_f, X_test, cat_feats)

    params = build_final_xgb_params(best_params)
    model = xgb.XGBClassifier(**params)

    t0 = time.perf_counter()
    model.fit(
        X_tr_f, y_tr_f,
        eval_set=[(X_val_f, y_val_f)],
        verbose=False,
    )
    train_time = time.perf_counter() - t0

    y_proba = model.predict_proba(X_test_al)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)
    metrics = compute_metrics(y_test.values, y_pred, y_proba)

    # Sauvegarde
    save_path = MODELS_DIR / f'xgboost_optuna_{tag}.ubj'
    model.save_model(str(save_path))

    row = {
        'Model':        f'XGBoost_Optuna_{tag}',
        'Train_File':   train_path.name,
        'N_Train':      len(X_full),
        'Fraud_Rate_Train_%': round(y_full.mean() * 100, 4),
        **{k: round(v, 6) if isinstance(v, float) else v for k, v in metrics.items()},
        'Best_Iteration': int(getattr(model, 'best_iteration', FINAL_N_ESTIMATORS)),
        'Training_Time_Seconds': round(train_time, 2),
        'Model_Path':   str(save_path),
    }

    del model, X_full, y_full, X_tr_f, X_val_f, y_tr_f, y_val_f, y_pred, y_proba
    gc.collect()
    return row


best_params = study.best_params
print('=== Best params from Optuna ===')
for k, v in best_params.items():
    print(f'  {k:<22} = {v}')

In [ ]:
results_rows = []

print('\n=== 1/2 Entraînement sur prepared_train_005.0_pct.parquet ===')
row_005 = train_final_model(DATA_DIR / TUNING_DATA, best_params, tag='005')
print(f"  F1={row_005['F1']:.4f} | PR-AUC={row_005['PR-AUC']:.4f} | "
      f"Recall={row_005['Recall']:.4f} | Precision={row_005['Precision']:.4f} | "
      f"best_iter={row_005['Best_Iteration']} | time={row_005['Training_Time_Seconds']:.1f}s")
results_rows.append(row_005)

print('\n=== 2/2 Entraînement sur prepared_train_original.parquet ===')
row_orig = train_final_model(DATA_DIR / PREPARED_ORIGINAL, best_params, tag='original')
print(f"  F1={row_orig['F1']:.4f} | PR-AUC={row_orig['PR-AUC']:.4f} | "
      f"Recall={row_orig['Recall']:.4f} | Precision={row_orig['Precision']:.4f} | "
      f"best_iter={row_orig['Best_Iteration']} | time={row_orig['Training_Time_Seconds']:.1f}s")
results_rows.append(row_orig)

---
## 10. ÉTAPE 4 — Logging & sauvegarde finale

In [ ]:
final_df = pd.DataFrame(results_rows)

# On ajoute une colonne avec les best_params en JSON pour la traçabilité
final_df['Best_Params_JSON'] = json.dumps(best_params)
final_df['Optuna_Best_Value_Val'] = round(study.best_value, 6)
final_df['Optuna_N_Trials']       = len(study.trials)
final_df['Optimize_Metric']       = OPTIMIZE_METRIC

final_df.to_csv(RESULTS_CSV, index=False)

print(f'Résultats sauvegardés : {RESULTS_CSV.resolve()}\n')

display_cols = ['Model', 'Train_File', 'N_Train', 'Fraud_Rate_Train_%',
                'F1', 'Recall', 'Precision', 'PR-AUC', 'ROC-AUC',
                'TP', 'FP', 'FN', 'TN', 'Best_Iteration', 'Training_Time_Seconds']
print(final_df[display_cols].to_string(index=False))

---
## 11. Comparaison visuelle des 2 modèles finaux

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ['F1', 'Recall', 'Precision', 'PR-AUC', 'ROC-AUC']
x = np.arange(len(metrics_to_plot))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, [row_005[m]  for m in metrics_to_plot], w, label='Trained on 5% undersample', color='#3498DB')
ax.bar(x + w/2, [row_orig[m] for m in metrics_to_plot], w, label='Trained on original distribution', color='#E74C3C')
ax.set_xticks(x); ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
ax.set_title('XGBoost Optuna — comparaison sur le test fixe (prepared_test_050.0_pct)')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
for i, m in enumerate(metrics_to_plot):
    ax.text(i - w/2, row_005[m]  + 0.02, f'{row_005[m]:.3f}',  ha='center', fontsize=9)
    ax.text(i + w/2, row_orig[m] + 0.02, f'{row_orig[m]:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(LOG_DIR / 'xgboost_optuna_comparison.png', dpi=110, bbox_inches='tight')
plt.show()

# Matrices de confusion
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, row, title in zip(axes, [row_005, row_orig],
                          ['Trained on 5% undersample', 'Trained on original distribution']):
    cm = np.array([[row['TN'], row['FP']], [row['FN'], row['TP']]])
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f'{v:,}', ha='center', va='center',
                color='white' if v > cm.max()/2 else 'black', fontsize=11)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['True 0', 'True 1'])
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(LOG_DIR / 'xgboost_optuna_confusion_matrices.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 12. Bilan

### Artefacts générés

| Fichier | Description |
|---|---|
| `prepared_train_original.parquet` | Dataset original, préparé avec le pipeline standard |
| `logs_xgboost_optuna_results.csv` | Résultats des 2 modèles finaux + best_params |
| `logs_training/models/xgboost_optuna_005.ubj` | Modèle entraîné sur 5% undersample |
| `logs_training/models/xgboost_optuna_original.ubj` | Modèle entraîné sur distribution originale |
| `logs_training/optuna/xgboost_study.pkl` | Étude Optuna rechargeable via `joblib.load` |
| `logs_training/optuna/trials.csv` | Tous les trials en tableur |
| `logs_training/optuna/*.html` | Visualisations interactives Optuna |
| `logs_training/xgboost_optuna_*.png` | Comparaison des 2 modèles + matrices de confusion |

### Recharger un modèle entraîné

```python
import xgboost as xgb
model = xgb.XGBClassifier()
model.load_model('logs_training/models/xgboost_optuna_original.ubj')
y_proba = model.predict_proba(X_new)[:, 1]
```

### Pistes d'amélioration possibles

1. **Optimisation du seuil de décision** (actuellement fixé à 0.5). Pour un modèle à fort `scale_pos_weight`, le seuil optimal est souvent bien différent → on peut gagner plusieurs points de F1.
2. **Calibration des probabilités** (`CalibratedClassifierCV` ou isotonic) si on veut exploiter les probas pour un downstream métier.
3. **Stacking** XGBoost + CatBoost via LogisticRegression pour bénéficier d'éventuelles erreurs décorrélées.
4. **Analyse SHAP** du meilleur modèle pour vérifier que les features dominantes sont bien les features généralisables (velocity, ratios, is_out_of_state, MCC).